In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler 

### Preparando os dados

In [2]:
df = pd.read_csv("../../0. Dados/telco_dataset.csv")
df.drop(['customerID'],axis=1, inplace=True)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df['TotalCharges'] = df['TotalCharges'].replace(" ", 0)

In [4]:
lista_X_quanti = ["tenure",
                  "MonthlyCharges",
                  "TotalCharges"]
lista_X_quali = ['OnlineSecurity',
                 'TechSupport']

In [5]:
for col in lista_X_quali:
    try:
        df[col] = df[col].astype("str")
        df[col] = df[col].str.strip()
    except:
        print(col)

for col in lista_X_quanti:
    try:df[col] = df[col].astype("float")
    except:print(col)

In [6]:
y = df['Churn'].replace(to_replace = ['Yes','No'],value = [1,0])
X = df[lista_X_quanti + lista_X_quali]

C:\Users\Carolina\AppData\Local\Temp\ipykernel_20304\4212662862.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df['Churn'].replace(to_replace = ['Yes','No'],value = [1,0])


### Transformer

In [7]:
# Podemos criar nossos próprios estimadores e transformers
class ColumnSelector(BaseEstimator, TransformerMixin):
    '''Seleciona um subset de um dado dataframe a partir de uma lista de colunas'''
    def __init__(self, cols_list):
        self.cols_list = cols_list
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        return X.loc[:, self.cols_list]

In [8]:
dados = ColumnSelector(lista_X_quanti+lista_X_quali)
dados.transform(df).head()

,tenure,MonthlyCharges,TotalCharges,OnlineSecurity,TechSupport
0,1.0,29.85,29.85,No,No
1,34.0,56.95,1889.50,Yes,No
2,2.0,53.85,108.15,Yes,No
3,45.0,42.30,1840.75,Yes,Yes
4,2.0,70.70,151.65,No,No


### Pipeline

In [9]:
num_pipe = Pipeline([('get_num_cols', ColumnSelector(lista_X_quanti)),
                ('fix_nan', SimpleImputer(missing_values=np.nan, strategy='median')),
                ('scale_data', MinMaxScaler())
])

In [10]:
dados_transformados = pd.DataFrame(num_pipe.fit_transform(df))
dados_transformados.head()

,0,1,2
0,0.013889,0.115423,0.003437
1,0.472222,0.385075,0.217564
2,0.027778,0.354229,0.012453
3,0.625000,0.239303,0.211951
4,0.027778,0.521891,0.017462


### ColumnTransformer

In [11]:
FeatureEng = ColumnTransformer(
    transformers=[
        ('cat_ohe', OneHotEncoder(), lista_X_quali),
        ('num_pipe', num_pipe, lista_X_quanti)
    ]
)
pd.DataFrame(FeatureEng.fit_transform(df))

,0,1,2,3,4,5,6,7,8
0,1.0,0.0,0.0,1.0,0.0,0.0,0.013889,0.115423,0.003437
1,0.0,0.0,1.0,1.0,0.0,0.0,0.472222,0.385075,0.217564
2,0.0,0.0,1.0,1.0,0.0,0.0,0.027778,0.354229,0.012453
3,0.0,0.0,1.0,0.0,0.0,1.0,0.625000,0.239303,0.211951
4,1.0,0.0,0.0,1.0,0.0,0.0,0.027778,0.521891,0.017462
...,...,...,...,...,...,...,...,...,...
7038,0.0,0.0,1.0,0.0,0.0,1.0,0.333333,0.662189,0.229194
7039,1.0,0.0,0.0,1.0,0.0,0.0,1.000000,0.845274,0.847792
7040,0.0,0.0,1.0,1.0,0.0,0.0,0.152778,0.112935,0.039892
7041,1.0,0.0,0.0,1.0,0.0,0.0,0.055556,0.558706,0.035303
